# 01 Peaks - Exploratory Data Analysis

This notebook will analyse called peaks.

## 01.1 Initialization

Load programs and data needed. Use calles peaks in source_data/cpm_stringent_peaks

In [2]:
docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -e MPLCONFIGDIR=/home/dalbao/.config/matplotlib \
        -v /tmp:/tmp \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

# Define software to use:
## bedtools for bed file manipulation
bedtools() {
    docker_run staphb/bedtools:2.31.1 bedtools "$@"
}
bedtools --version

# Function: bed_length_stats: compute length statistics for regions in a BED file
# Usage: bed_length_stats <input.bed>
# Assumes standard BED format (chrom, start, end, ...), 0-based half-open coords.
# Length of each region = end - start.
bed_length_stats() {
    local bed="$1"

    if [[ -z "$bed" || ! -f "$bed" ]]; then
        echo "Usage: bed_length_stats <input.bed>" >&2
        return 1
    fi

    awk '
    # Skip header/track/comment lines commonly found in BED files
    /^(#|track|browser)/ { next }
    NF < 3 { next }
    {
        len = $3 - $2
        if (len < 0) next          # skip malformed intervals
        lengths[n++] = len
        sum += len
    }
    END {
        if (n == 0) {
            print "No valid regions found." > "/dev/stderr"
            exit 1
        }

        # Sort lengths in-place (simple insertion sort; fine for typical BED sizes,
        # swap for a faster sort if you have millions of regions)
        for (i = 1; i < n; i++) {
            key = lengths[i]
            j = i - 1
            while (j >= 0 && lengths[j] > key) {
                lengths[j+1] = lengths[j]
                j--
            }
            lengths[j+1] = key
        }

        mean = sum / n

        # Percentile via linear interpolation on the sorted array (0-indexed)
        # p is a fraction between 0 and 1
        # rank = p * (n - 1)
        printf "n_regions\t%d\n", n
        printf "mean\t%.2f\n", mean
        printf "median\t%.2f\n", pct(0.50)
        printf "Q10\t%.2f\n", pct(0.10)
        printf "Q25\t%.2f\n", pct(0.25)
        printf "Q50\t%.2f\n", pct(0.50)
        printf "Q75\t%.2f\n", pct(0.75)
        printf "Q90\t%.2f\n", pct(0.90)
    }

    function pct(p,    rank, lo, hi, frac) {
        rank = p * (n - 1)
        lo = int(rank)
        hi = (lo + 1 < n) ? lo + 1 : lo
        frac = rank - lo
        return lengths[lo] + frac * (lengths[hi] - lengths[lo])
    }
    ' "$bed"
}

cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun
mkdir -p 01_peakEDA

ls -1 source_data/cpm_stringent_peaks

plotHeatmap 3.5.6
bedtools v2.31.1
early_Runx1.CPM.stringent.bed
early_Runx3.CPM.stringent.bed
late_Runx1.CPM.stringent.bed
late_Runx3.CPM.stringent.bed
memory_Runx1.CPM.stringent.bed
memory_Runx3.CPM.stringent.bed
shCd19_Runx1.CPM.stringent.bed
shCd19_Runx3.CPM.stringent.bed
shRunx3_Runx1.CPM.stringent.bed
shRunx3_Runx3.CPM.stringent.bed
terminal_Runx1.CPM.stringent.bed
terminal_Runx3.CPM.stringent.bed


Check contents of bed files:

In [3]:
# Check contents of bed files
head -n 1 source_data/cpm_stringent_peaks/*.bed

==> source_data/cpm_stringent_peaks/early_Runx1.CPM.stringent.bed <==
6	70915050	70918550	1687.6	1.10061	6:70917450-70917500

==> source_data/cpm_stringent_peaks/early_Runx3.CPM.stringent.bed <==
1	4158350	4160350	885.632	1.05747	1:4159100-4159300

==> source_data/cpm_stringent_peaks/late_Runx1.CPM.stringent.bed <==

==> source_data/cpm_stringent_peaks/late_Runx3.CPM.stringent.bed <==
1	3304750	3306500	1047.67	1.13262	1:3306050-3306450

==> source_data/cpm_stringent_peaks/memory_Runx1.CPM.stringent.bed <==
1	6319500	6321300	842.281	1.25446	1:6320250-6320300

==> source_data/cpm_stringent_peaks/memory_Runx3.CPM.stringent.bed <==
1	3459650	3461900	1351.1	1.0528	1:3460900-3461300

==> source_data/cpm_stringent_peaks/shCd19_Runx1.CPM.stringent.bed <==
1	3203900	3205200	638.098	0.739823	1:3204200-3205200

==> source_data/cpm_stringent_peaks/shCd19_Runx3.CPM.stringent.bed <==
1	10037650	10038650	3383.86	9.20547	1:10038150-10038200

==> source_data/cpm_stringent_peaks/shRunx3_Runx1.CPM.string

Add a unique identifier to each peak

In [4]:
outdir=01_peakEDA/named_peaks
mkdir -p "$outdir"

for bed in source_data/cpm_stringent_peaks/*.CPM.stringent.bed; do
    prefix=$(basename "$bed" .CPM.stringent.bed)

    # Rebuild the line: chrom, start, end, ID, then the original cols 4-6
    awk -v OFS='\t' -v p="$prefix" \
        '{ print $1, $2, $3, p "_" NR, $4, $5, $6 }' \
        "$bed" > "$outdir/${prefix}.CPM.stringent.named.bed"
done

head -n 1 01_peakEDA/named_peaks/*.bed

==> 01_peakEDA/named_peaks/early_Runx1.CPM.stringent.named.bed <==
6	70915050	70918550	early_Runx1_1	1687.6	1.10061	6:70917450-70917500

==> 01_peakEDA/named_peaks/early_Runx3.CPM.stringent.named.bed <==
1	4158350	4160350	early_Runx3_1	885.632	1.05747	1:4159100-4159300

==> 01_peakEDA/named_peaks/late_Runx1.CPM.stringent.named.bed <==

==> 01_peakEDA/named_peaks/late_Runx3.CPM.stringent.named.bed <==
1	3304750	3306500	late_Runx3_1	1047.67	1.13262	1:3306050-3306450

==> 01_peakEDA/named_peaks/memory_Runx1.CPM.stringent.named.bed <==
1	6319500	6321300	memory_Runx1_1	842.281	1.25446	1:6320250-6320300

==> 01_peakEDA/named_peaks/memory_Runx3.CPM.stringent.named.bed <==
1	3459650	3461900	memory_Runx3_1	1351.1	1.0528	1:3460900-3461300

==> 01_peakEDA/named_peaks/shCd19_Runx1.CPM.stringent.named.bed <==
1	3203900	3205200	shCd19_Runx1_1	638.098	0.739823	1:3204200-3205200

==> 01_peakEDA/named_peaks/shCd19_Runx3.CPM.stringent.named.bed <==
1	10037650	10038650	shCd19_Runx3_1	3383.86	9.20547	1:10

#### Extract maximum signal regions

In [5]:
OUTDIR="01_peakEDA/max_signal"
INDIR="01_peakEDA/named_peaks"

mkdir -p "$OUTDIR"

shopt -s nullglob
bed_files=("$INDIR"/*.bed)

for f in "${bed_files[@]}"; do
    base=$(basename "$f")
    out="$OUTDIR/${base%.bed}.maxsignal.bed"

    awk 'BEGIN{OFS="\t"}
        {
            # column 7: chr:start-end
            n = split($7, region, "[:-]")
            if (n != 3) {
                print "Skipping malformed line in " FILENAME ": " $0 > "/dev/stderr"
                next
            }
            print region[1], region[2], region[3], $4
        }' "$f" > "$out"
done

head -n 1 01_peakEDA/max_signal/*.bed

==> 01_peakEDA/max_signal/early_Runx1.CPM.stringent.named.maxsignal.bed <==
6	70917450	70917500	early_Runx1_1

==> 01_peakEDA/max_signal/early_Runx3.CPM.stringent.named.maxsignal.bed <==
1	4159100	4159300	early_Runx3_1

==> 01_peakEDA/max_signal/late_Runx1.CPM.stringent.named.maxsignal.bed <==

==> 01_peakEDA/max_signal/late_Runx3.CPM.stringent.named.maxsignal.bed <==
1	3306050	3306450	late_Runx3_1

==> 01_peakEDA/max_signal/memory_Runx1.CPM.stringent.named.maxsignal.bed <==
1	6320250	6320300	memory_Runx1_1

==> 01_peakEDA/max_signal/memory_Runx3.CPM.stringent.named.maxsignal.bed <==
1	3460900	3461300	memory_Runx3_1

==> 01_peakEDA/max_signal/shCd19_Runx1.CPM.stringent.named.maxsignal.bed <==
1	3204200	3205200	shCd19_Runx1_1

==> 01_peakEDA/max_signal/shCd19_Runx3.CPM.stringent.named.maxsignal.bed <==
1	10038150	10038200	shCd19_Runx3_1

==> 01_peakEDA/max_signal/shRunx3_Runx1.CPM.stringent.named.maxsignal.bed <==
1	3694050	3694400	shRunx3_Runx1_1

==> 01_peakEDA/max_signal/shRunx3_Runx

## 01.2 Initial Heatmaps
Do heatmaps of peaks to analyze. Frist make associative array of files for easy filename handling.

In [6]:
# First make a associative array to store bigWig and peak locations for each sample
declare -A bigWigFiles
declare -A rawPeaks

for group in shCd19 shRunx3 memory early late terminal; do
    for target in Runx3 Runx1; do
        fn=${group}_${target}_log2.bigWig
        fpath="source_data/bg_corrected_bigWigs/"
        bigWigFiles[${group}_${target}]="${fpath}${fn}"

        fn=${group}_${target}.CPM.stringent.named.maxsignal.bed
        fpath="01_peakEDA/max_signal/"
        rawPeaks[${group}_${target}]="${fpath}${fn}"
    done
done

echo Sample: ${bigWigFiles["shCd19_Runx3"]}
echo Sample: ${rawPeaks["shCd19_Runx3"]}

Sample: source_data/bg_corrected_bigWigs/shCd19_Runx3_log2.bigWig
Sample: 01_peakEDA/max_signal/shCd19_Runx3.CPM.stringent.named.maxsignal.bed


Compute matrix for perturbation experiment.

In [7]:
mkdir -p 01_peakEDA/02_initialHeatmaps

deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    -R \
    ${rawPeaks["shCd19_Runx3"]} \
    ${rawPeaks["shRunx3_Runx3"]} \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 01_peakEDA/02_initialHeatmaps/perturbSamples.Runx3target-day5peaks.gz

deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    -R \
    ${rawPeaks["shCd19_Runx1"]} \
    ${rawPeaks["shRunx3_Runx1"]} \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 01_peakEDA/02_initialHeatmaps/perturbSamples.Runx1target-day5peaks.gz

Plot heatmaps

In [8]:
deeptools plotHeatmap \
    -m "01_peakEDA/02_initialHeatmaps/perturbSamples.Runx3target-day5peaks.gz" \
    -out "01_peakEDA/02_initialHeatmaps/perturbSamples.Runx3target-day5peaks.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "shCd19_Runx3" "shRunx3_Runx3" \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1"

deeptools plotHeatmap \
    -m "01_peakEDA/02_initialHeatmaps/perturbSamples.Runx1target-day5peaks.gz" \
    -out "01_peakEDA/02_initialHeatmaps/perturbSamples.Runx1target-day5peaks.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "shCd19_Runx1" "shRunx3_Runx1" \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1"

ls -1 01_peakEDA/02_initialHeatmaps

perturbSamples.Runx1target-day5peaks.gz
perturbSamples.Runx1target-day5peaks.pdf
perturbSamples.Runx3target-day5peaks.gz
perturbSamples.Runx3target-day5peaks.pdf


Re-compute day 5 peaks by clustering.

In [9]:
mkdir -p 01_peakEDA/02_initialHeatmaps/day5peaks_clustering

for k in 3 4 5 6; do

    deeptools plotHeatmap \
        -m "01_peakEDA/02_initialHeatmaps/perturbSamples.Runx3target-day5peaks.gz" \
        -out "01_peakEDA/02_initialHeatmaps/day5peaks_clustering/perturbSamples.Runx3target-day5peaks.k${k}.pdf" \
        --kmeans ${k} \
        --sortUsing sum \
        --colorMap "RdYlBu_r" \
        --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
        --outFileSortedRegions "01_peakEDA/02_initialHeatmaps/day5peaks_clustering/perturbSamples.Runx3target-day5peaks.k${k}.sorted.bed"

    deeptools plotHeatmap \
        -m "01_peakEDA/02_initialHeatmaps/perturbSamples.Runx1target-day5peaks.gz" \
        -out "01_peakEDA/02_initialHeatmaps/day5peaks_clustering/perturbSamples.Runx1target-day5peaks.k${k}.pdf" \
        --kmeans ${k} \
        --sortUsing sum \
        --colorMap "RdYlBu_r" \
        --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
        --outFileSortedRegions "01_peakEDA/02_initialHeatmaps/day5peaks_clustering/perturbSamples.Runx1target-day5peaks.k${k}.sorted.bed"

done
ls -1 01_peakEDA/02_initialHeatmaps/day5peaks_clustering

*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
perturbSamples.Runx1target-day5peaks.k3.pdf
perturbSamples.Runx1target-day5peaks.k3.sorted.bed
perturbSamples.Runx1target-day5peaks.k4.pdf
perturbSamples.Runx1target-day5peaks.k4.sorted.bed
perturbSamples.Runx1target-day5peaks.k5.pdf
perturbSamples.Runx1target-day5peaks.k5.sorted.bed
perturbSamples.Runx1target-day5peaks.k6.pdf
perturbSamples.Runx1target-day5peaks.k6.sorted.bed
perturbSamples.Runx3target-day5peaks.k3.pdf
perturbSamples.Runx3target-day5peaks.k3.sorted.b

Do day 8 natural samples

In [10]:
mkdir -p 01_peakEDA/02_initialHeatmaps

deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    ${bigWigFiles["memory_Runx3"]} \
    ${bigWigFiles["early_Runx3"]} \
    ${bigWigFiles["late_Runx3"]} \
    ${bigWigFiles["terminal_Runx3"]} \
    ${bigWigFiles["memory_Runx1"]} \
    ${bigWigFiles["early_Runx1"]} \
    ${bigWigFiles["late_Runx1"]} \
    ${bigWigFiles["terminal_Runx1"]} \
    -R \
    ${rawPeaks["memory_Runx3"]} \
    ${rawPeaks["early_Runx3"]} \
    ${rawPeaks["late_Runx3"]} \
    ${rawPeaks["terminal_Runx3"]} \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 01_peakEDA/02_initialHeatmaps/naturalSamples.Runx3target-day8peaks.gz

deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    ${bigWigFiles["memory_Runx3"]} \
    ${bigWigFiles["early_Runx3"]} \
    ${bigWigFiles["late_Runx3"]} \
    ${bigWigFiles["terminal_Runx3"]} \
    ${bigWigFiles["memory_Runx1"]} \
    ${bigWigFiles["early_Runx1"]} \
    ${bigWigFiles["late_Runx1"]} \
    ${bigWigFiles["terminal_Runx1"]} \
    -R \
    ${rawPeaks["memory_Runx1"]} \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 01_peakEDA/02_initialHeatmaps/naturalSamples.Runx1target-day8peaks.gz

ls -1 01_peakEDA/02_initialHeatmaps/*day8peaks.gz


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698

The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
01_peakEDA/02_initialHeatmaps/naturalSamples.Runx1target-day8peaks.gz
01_peakEDA/02_initialHeatmaps/naturalSamples.Runx3target-day8peaks.gz


Plot heatmaps

In [11]:
deeptools plotHeatmap \
    -m "01_peakEDA/02_initialHeatmaps/naturalSamples.Runx3target-day8peaks.gz" \
    -out "01_peakEDA/02_initialHeatmaps/naturalSamples.Runx3target-day8peaks.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                   "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                   "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1"

deeptools plotHeatmap \
    -m "01_peakEDA/02_initialHeatmaps/naturalSamples.Runx1target-day8peaks.gz" \
    -out "01_peakEDA/02_initialHeatmaps/naturalSamples.Runx1target-day8peaks.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "memory_Runx1" \
    --samplesLabel  "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                   "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                   "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1"

ls -1 01_peakEDA/02_initialHeatmaps/*day8peaks.pdf

01_peakEDA/02_initialHeatmaps/naturalSamples.Runx1target-day8peaks.pdf
01_peakEDA/02_initialHeatmaps/naturalSamples.Runx3target-day8peaks.pdf


In [12]:
mkdir -p 01_peakEDA/02_initialHeatmaps/day8peaks_clustering

for k in 3 4 5 6 7; do

    deeptools plotHeatmap \
        -m "01_peakEDA/02_initialHeatmaps/naturalSamples.Runx3target-day8peaks.gz" \
        -out "01_peakEDA/02_initialHeatmaps/day8peaks_clustering/naturalSamples.Runx3target-day8peaks.k${k}.pdf" \
        --kmeans ${k} \
        --sortUsing sum \
        --colorMap "RdYlBu_r" \
        --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                        "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                        "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
        --outFileSortedRegions "01_peakEDA/02_initialHeatmaps/day8peaks_clustering/naturalSamples.Runx3target-day8peaks.k${k}.sorted.bed"

    deeptools plotHeatmap \
        -m "01_peakEDA/02_initialHeatmaps/naturalSamples.Runx1target-day8peaks.gz" \
        -out "01_peakEDA/02_initialHeatmaps/day8peaks_clustering/naturalSamples.Runx1target-day8peaks.k${k}.pdf" \
        --kmeans ${k} \
        --sortUsing sum \
        --colorMap "RdYlBu_r" \
        --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                        "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                        "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
        --outFileSortedRegions "01_peakEDA/02_initialHeatmaps/day8peaks_clustering/naturalSamples.Runx1target-day8peaks.k${k}.sorted.bed"

done

ls -1 01_peakEDA/02_initialHeatmaps/day8peaks_clustering/*

*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
01_peakEDA/02_initialHeatmaps/day8peaks_clustering/naturalSamples.Runx1target-day8peaks.k3.pdf
01_peakEDA/02_initialHeatmaps/day8peaks_clustering/naturalSamples.Runx1target-day8peaks.k3.sorted.bed
01_peakEDA/02_initialHeatmaps/day8peaks_clustering/naturalSamples.Runx1target-day8peaks.k4.pdf
01_peakEDA/02_initialHeatmaps/day8peaks_clusteri

# 01.3 Peak Cleanliness Analysis

Cleanup day 8 RUNX3 peaks using heatmaps.

In [13]:
for group in memory early late terminal; do
    mkdir -p 01_peakEDA/03_peakCleanup/${group}

    deeptools computeMatrix reference-point \
        -S \
        ${bigWigFiles["shCd19_Runx3"]} \
        ${bigWigFiles["shRunx3_Runx3"]} \
        ${bigWigFiles["shCd19_Runx1"]} \
        ${bigWigFiles["shRunx3_Runx1"]} \
        ${bigWigFiles["memory_Runx3"]} \
        ${bigWigFiles["early_Runx3"]} \
        ${bigWigFiles["late_Runx3"]} \
        ${bigWigFiles["terminal_Runx3"]} \
        ${bigWigFiles["memory_Runx1"]} \
        ${bigWigFiles["early_Runx1"]} \
        ${bigWigFiles["late_Runx1"]} \
        ${bigWigFiles["terminal_Runx1"]} \
        -R \
        ${rawPeaks["${group}_Runx3"]} \
        --referencePoint center \
        -b 1500 -a 1500 \
        --numberOfProcessors 36 \
        --sortUsing mean \
        -out 01_peakEDA/03_peakCleanup/${group}/Runx3-${group}.gz

    for k in 3 4 5 6; do

        deeptools plotHeatmap \
            -m "01_peakEDA/03_peakCleanup/${group}/Runx3-${group}.gz" \
            -out "01_peakEDA/03_peakCleanup/${group}/Runx3-${group}.k${k}.pdf" \
            --kmeans ${k} \
            --sortUsing sum \
            --colorMap "RdYlBu_r" \
            --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                            "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                            "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
            --outFileSortedRegions "01_peakEDA/03_peakCleanup/${group}/Runx3-${group}.k${k}.sorted.bed"

    done

    ls -1 01_peakEDA/03_peakCleanup/${group}/*

done


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
01_peakEDA/03_peakCleanup/memory/Runx3-memory.gz
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k3.pdf
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k3.sorted.bed
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k4.pdf
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k4.sorted.bed
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k5.pdf
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k5.sorted.bed
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k6.pdf
01_peakEDA/03_peakCleanup/memory/Runx3-memory.k6.sorted.bed

The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
01_

# 01.4 Peak Cleanup and Consolidate

Isolate good day 5 max signal peaks

In [14]:
mkdir -p 01_peakEDA/04_peakConsolidate/maxsignal

# Do Runx3 first
cat 01_peakEDA/02_initialHeatmaps/day5peaks_clustering/perturbSamples.Runx3target-day5peaks.k4.sorted.bed \
    | grep "cluster_1\|cluster_2\|cluster_3" \
    > 01_peakEDA/04_peakConsolidate/maxsignal/Runx3.good.maxsignal.bed
# Replace cluster_1 with Runx3C1, cluster_2 with Runx3C2 and cluster_3 with Runx3C3
sed -i 's/cluster_1/Runx3C1/g' 01_peakEDA/04_peakConsolidate/maxsignal/Runx3.good.maxsignal.bed
sed -i 's/cluster_2/Runx3C2/g' 01_peakEDA/04_peakConsolidate/maxsignal/Runx3.good.maxsignal.bed
sed -i 's/cluster_3/Runx3C3/g' 01_peakEDA/04_peakConsolidate/maxsignal/Runx3.good.maxsignal.bed

# cluster_1, cluster_2 of Runx1 k4 beds are good.
cat 01_peakEDA/02_initialHeatmaps/day5peaks_clustering/perturbSamples.Runx1target-day5peaks.k4.sorted.bed \
    | grep "cluster_1\|cluster_2" \
    > 01_peakEDA/04_peakConsolidate/maxsignal/Runx1.good.maxsignal.bed
# Replace cluster_1 with Runx1C1 and cluster_2 with Runx1C2
sed -i 's/cluster_1/Runx1C1/g' 01_peakEDA/04_peakConsolidate/maxsignal/Runx1.good.maxsignal.bed
sed -i 's/cluster_2/Runx1C2/g' 01_peakEDA/04_peakConsolidate/maxsignal/Runx1.good.maxsignal.bed

head -n 5 01_peakEDA/04_peakConsolidate/maxsignal/Runx*.bed

==> 01_peakEDA/04_peakConsolidate/maxsignal/Runx1.good.maxsignal.bed <==
12	98269900	98270400	12:98269900-98270400	.	.	98269900	98270400	0	1	500	98269891	Runx1C1
12	98270200	98270450	12:98270200-98270450	.	.	98270200	98270450	0	1	250	98270191	Runx1C1
4	32250300	32250350	4:32250300-32250350	.	.	32250300	32250350	0	1	50	32250297	Runx1C1
4	32250650	32250700	4:32250650-32250700	.	.	32250650	32250700	0	1	50	32250647	Runx1C1
14	54223900	54223950	14:54223900-54223950	.	.	54223900	54223950	0	1	50	54223895	Runx1C1

==> 01_peakEDA/04_peakConsolidate/maxsignal/Runx3.good.maxsignal.bed <==
12	98270250	98270300	12:98270250-98270300	.	.	98270250	98270300	0	1	50	98270241	Runx3C1
12	98270400	98270450	12:98270400-98270450	.	.	98270400	98270450	0	1	50	98270391	Runx3C1
4	32250700	32250750	4:32250700-32250750	.	.	32250700	32250750	0	1	50	32250697	Runx3C1
4	32250650	32250700	4:32250650-32250700	.	.	32250650	32250700	0	1	50	32250647	Runx3C1
6	50417250	50417350	6:50417250-50417350	.	.	50417250	50417350	0	1	1

Isolate good day 8 max signal peaks

In [15]:
# In the folder 01_peakEDA/03_peakCleanup are four folders:
# memory early late terminal
# Each folder has a file named Runx3-group.k3.sorted.bed
# Wherein group is the name of the folder (Runx3-early.k3.sorted.bed, Runx3-late.k3.sorted.bed...)
# Write a script that reads each file
# Takes only lines with cluster_1
# And outputs into the new file 04_peakConsolidate/maxsignal/group.good.maxsignal.bed

for group in memory early late terminal; do
    input_file="01_peakEDA/03_peakCleanup/${group}/Runx3-${group}.k3.sorted.bed"
    output_file="01_peakEDA/04_peakConsolidate/maxsignal/${group}.good.maxsignal.bed"
    
    # Extract lines with cluster_1 and write to the output file
    grep "cluster_1" "$input_file" > "$output_file"
    
    # Optionally, you can replace cluster_1 with a more descriptive name if needed
    sed -i "s/cluster_1/${group}C1/g" "$output_file"

    echo "==> ${output_file} <=="
    head -n 5 "$output_file"

done

==> 01_peakEDA/04_peakConsolidate/maxsignal/memory.good.maxsignal.bed <==
1	64532300	64532450	1:64532300-64532450	.	.	64532300	64532450	0	1	150	64532294	memoryC1
8	25274650	25274700	8:25274650-25274700	.	.	25274650	25274700	0	1	50	25274648	memoryC1
9	57339850	57340000	9:57339850-57340000	.	.	57339850	57340000	0	1	150	57339845	memoryC1
2	114145200	114145250	2:114145200-114145250	.	.	114145200	114145250	0	1	50	114145199	memoryC1
9	57076150	57076200	9:57076150-57076200	.	.	57076150	57076200	0	1	50	57076145	memoryC1
==> 01_peakEDA/04_peakConsolidate/maxsignal/early.good.maxsignal.bed <==
12	98270250	98270300	12:98270250-98270300	.	.	98270250	98270300	0	1	50	98270241	earlyC1
8	25274500	25274700	8:25274500-25274700	.	.	25274500	25274700	0	1	200	25274498	earlyC1
5	33658250	33658350	5:33658250-33658350	.	.	33658250	33658350	0	1	100	33658247	earlyC1
9	57339800	57340000	9:57339800-57340000	.	.	57339800	57340000	0	1	200	57339795	earlyC1
1	171629500	171629650	1:171629500-171629650	.	.	171629500	17

In [16]:
# Combine the bed files into combined with the header:
# #chrom	start	end	name	score	strand	thickStart	thickEnd	itemRGB	blockCount	blockSizes	blockStart	deepTools_group
echo '#chrom start end name score strand thickStart thickEnd itemRGB blockCount blockSizes blockStart deepTools_group' > 01_peakEDA/04_peakConsolidate/maxsignal/all_good.maxsignal.bed

cat 01_peakEDA/04_peakConsolidate/maxsignal/Runx3.good.maxsignal.bed \
    01_peakEDA/04_peakConsolidate/maxsignal/Runx1.good.maxsignal.bed \
    01_peakEDA/04_peakConsolidate/maxsignal/memory.good.maxsignal.bed \
    01_peakEDA/04_peakConsolidate/maxsignal/early.good.maxsignal.bed \
    01_peakEDA/04_peakConsolidate/maxsignal/late.good.maxsignal.bed \
    01_peakEDA/04_peakConsolidate/maxsignal/terminal.good.maxsignal.bed \
    >> 01_peakEDA/04_peakConsolidate/maxsignal/all_good.maxsignal.bed

In [17]:
deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    ${bigWigFiles["memory_Runx3"]} \
    ${bigWigFiles["early_Runx3"]} \
    ${bigWigFiles["late_Runx3"]} \
    ${bigWigFiles["terminal_Runx3"]} \
    ${bigWigFiles["memory_Runx1"]} \
    ${bigWigFiles["early_Runx1"]} \
    ${bigWigFiles["late_Runx1"]} \
    ${bigWigFiles["terminal_Runx1"]} \
    -R \
    01_peakEDA/04_peakConsolidate/maxsignal/all_good.maxsignal.bed \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 01_peakEDA/04_peakConsolidate/maxsignal/all.gz

ls -1 01_peakEDA/04_peakConsolidate/maxsignal/all.gz


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
01_peakEDA/04_peakConsolidate/maxsignal/all.gz


In [18]:
deeptools plotHeatmap \
    -m "01_peakEDA/04_peakConsolidate/maxsignal/all.gz" \
    -out "01_peakEDA/04_peakConsolidate/maxsignal/all.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                    "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                    "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1"

ls -1 01_peakEDA/04_peakConsolidate/maxsignal/all.pdf

01_peakEDA/04_peakConsolidate/maxsignal/all.pdf


In [19]:
mkdir -p 01_peakEDA/04_peakConsolidate/maxsignal/kmeans
for k in 1 2 3 4 5 6 7; do

    deeptools plotHeatmap \
        -m "01_peakEDA/04_peakConsolidate/maxsignal/all.gz" \
        -out "01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k${k}.pdf" \
        --kmeans ${k} \
        --sortUsing sum \
        --colorMap "RdYlBu_r" \
        --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                        "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                        "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
        --outFileSortedRegions "01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k${k}.sorted.bed"

done
ls -1 01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k*.pdf

*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
*Warning* For clustering nan values have to be replaced by zeros 
01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k1.pdf
01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k2.pdf
01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k3.pdf
01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k4.pdf
01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k5.pdf
01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k6.pdf
01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k7.pdf


In [20]:
deeptools plotHeatmap \
    -m "01_peakEDA/04_peakConsolidate/maxsignal/all.gz" \
    -out "01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k2.pretty.pdf" \
    --kmeans 2 \
    --sortUsing sum \
    --colorMap RdYlBu_r RdYlBu_r RdYlGn_r RdYlGn_r RdYlBu_r RdYlBu_r RdYlBu_r RdYlBu_r RdYlGn_r RdYlGn_r RdYlGn_r RdYlGn_r \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                    "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                    "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
    --zMin 0 \
    --zMax 1.25 1.25 1.00 1.00 1.20 1.00 1.00 1.00 0.75 0.75 0.75 0.75 \
    --yMax 3.00 3.00 1.75 1.75 1.60 1.60 1.60 1.60 0.90 0.90 0.90 0.90

*Warning* For clustering nan values have to be replaced by zeros 


In [21]:
bed_length_stats 01_peakEDA/04_peakConsolidate/maxsignal/kmeans/all.k2.sorted.bed

n_regions	8802
mean	118.79
median	50.00
Q10	50.00
Q25	50.00
Q50	50.00
Q75	150.00
Q90	200.00


In [22]:
deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    ${bigWigFiles["memory_Runx3"]} \
    ${bigWigFiles["early_Runx3"]} \
    ${bigWigFiles["late_Runx3"]} \
    ${bigWigFiles["terminal_Runx3"]} \
    ${bigWigFiles["memory_Runx1"]} \
    ${bigWigFiles["early_Runx1"]} \
    ${bigWigFiles["late_Runx1"]} \
    ${bigWigFiles["terminal_Runx1"]} \
    -R \
    01_peakEDA/04_peakConsolidate/maxsignal.clean.bed \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 01_peakEDA/04_peakConsolidate/maxsignal.clean.gz



The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
Traceback (most recent call last):
  File "/usr/local/bin/computeMatrix", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/site-packages/deeptools/computeMatrix.py", line 406, in main
    hm.computeMatrix(scores_file_list, args.regionsFileName, parameters, blackListFileName=args.blackListFileName, verbose=args.verbose, allArgs=args)
  File "/usr/local/lib/python3.12/site-packages/deeptools/heatmapper.py", line 252, in computeMatrix
    res, labels = mapReduce.mapReduce([score_file_list, parameters],
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/deeptools/mapReduce.py", line 85, in mapReduce
    bed_interval_tree = GTF(bedFile, defaultGroup=defaultGroup, transcriptID=transcriptID, exonID=exonID, transcript_id_designator=transcript_id_designator, keepExons=k

: 1

In [ ]:
deeptools plotHeatmap \
    -m "01_peakEDA/04_peakConsolidate/maxsignal.clean.gz" \
    -out "01_peakEDA/04_peakConsolidate/maxsignal.clean.k2.pdf" \
    --kmeans 2 \
    --sortUsing sum \
    --colorMap RdYlBu_r RdYlBu_r RdYlGn_r RdYlGn_r RdYlBu_r RdYlBu_r RdYlBu_r RdYlBu_r RdYlGn_r RdYlGn_r RdYlGn_r RdYlGn_r \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                    "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                    "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
    --zMin 0 \
    --zMax 1.25 1.25 1.00 1.00 1.20 1.00 1.00 1.00 0.75 0.75 0.75 0.75 \
    --yMax 2.50 2.50 1.40 1.40 1.25 1.25 1.25 1.25 0.65 0.65 0.65 0.65 \
    --outFileSortedRegions "01_peakEDA/04_peakConsolidate/maxsignal.clean.clustered-k2.bed"

###  Full Peak Recovery and Consolidation to Consensus

In [23]:
mkdir -p 01_peakEDA/04_peakConsolidate/individual

# For all individual BED in 01_peakEDA/named_peaks
# Keep only rows wherein column  7 ($7) is in
# Column 4 ($4) of 01_peakEDA/04_peakConsolidate/maxsignal/all_good.maxsignal.bed

for bed in 01_peakEDA/named_peaks/*.bed; do
    prefix=$(basename "$bed" .bed)
    out="01_peakEDA/04_peakConsolidate/individual/${prefix}.good.bed"

    awk 'NR==FNR { ids[$4]; next } $7 in ids' \
        01_peakEDA/04_peakConsolidate/maxsignal/all_good.maxsignal.bed \
        "$bed" > "$out"

    echo "==> ${out} <=="
    head -n 2 "$out"
done

==> 01_peakEDA/04_peakConsolidate/individual/early_Runx1.CPM.stringent.named.good.bed <==
==> 01_peakEDA/04_peakConsolidate/individual/early_Runx3.CPM.stringent.named.good.bed <==
1	65176350	65179800	early_Runx3_130	1506.9	1.18965	1:65178550-65178700
1	91297200	91298650	early_Runx3_182	793.103	1.18965	1:91298300-91298400
==> 01_peakEDA/04_peakConsolidate/individual/late_Runx1.CPM.stringent.named.good.bed <==
==> 01_peakEDA/04_peakConsolidate/individual/late_Runx3.CPM.stringent.named.good.bed <==
1	13558750	13561100	late_Runx3_22	1651.74	1.51016	1:13559800-13559850
1	13592600	13594200	late_Runx3_23	906.097	1.32139	1:13593400-13593600
==> 01_peakEDA/04_peakConsolidate/individual/memory_Runx1.CPM.stringent.named.good.bed <==
12	31498550	31500100	memory_Runx1_265	1200.7	1.79209	12:31499650-31499800
4	32587100	32589050	memory_Runx1_865	896.045	1.07525	4:32587900-32588000
==> 01_peakEDA/04_peakConsolidate/individual/memory_Runx3.CPM.stringent.named.good.bed <==
1	64531350	64532850	memory_Run

In [24]:
# Compute length statistics for files in 01_peakEDA/04_peakConsolidate/individual/*.bed
for bed in 01_peakEDA/04_peakConsolidate/individual/*.bed; do
    echo "==> ${bed} <=="
    bed_length_stats "$bed"
done

==> 01_peakEDA/04_peakConsolidate/individual/early_Runx1.CPM.stringent.named.good.bed <==
No valid regions found.
==> 01_peakEDA/04_peakConsolidate/individual/early_Runx3.CPM.stringent.named.good.bed <==
n_regions	80
mean	1580.00
median	1500.00
Q10	950.00
Q25	1150.00
Q50	1500.00
Q75	1962.50
Q90	2350.00
==> 01_peakEDA/04_peakConsolidate/individual/late_Runx1.CPM.stringent.named.good.bed <==
No valid regions found.
==> 01_peakEDA/04_peakConsolidate/individual/late_Runx3.CPM.stringent.named.good.bed <==
n_regions	329
mean	1470.82
median	1450.00
Q10	900.00
Q25	1100.00
Q50	1450.00
Q75	1750.00
Q90	2150.00
==> 01_peakEDA/04_peakConsolidate/individual/memory_Runx1.CPM.stringent.named.good.bed <==
n_regions	2
mean	1750.00
median	1750.00
Q10	1590.00
Q25	1650.00
Q50	1750.00
Q75	1850.00
Q90	1910.00
==> 01_peakEDA/04_peakConsolidate/individual/memory_Runx3.CPM.stringent.named.good.bed <==
n_regions	98
mean	1346.43
median	1250.00
Q10	750.00
Q25	1000.00
Q50	1250.00
Q75	1650.00
Q90	1945.00
==> 01_peak

In [26]:
# Combine unique peaks into fullpeak.clean.bed

DIR=01_peakEDA/04_peakConsolidate/individual
OUT=01_peakEDA/04_peakConsolidate/fullpeak.clean.bed
WORK=01_peakEDA/04_peakConsolidate/.precedence_tmp
mkdir -p "$WORK" "$(dirname "$OUT")"

PRIORITY=(
  shCd19_Runx3
  shRunx3_Runx3
  shRunx3_Runx1
  shCd19_Runx1
  terminal_Runx3
  late_Runx3
  early_Runx3
  memory_Runx3
  terminal_Runx1
  late_Runx1
  early_Runx1
  memory_Runx1
)

: > "$WORK/consensus.bed"

for s in "${PRIORITY[@]}"; do
  f="$DIR/${s}.CPM.stringent.named.good.bed"
  [[ -s $f ]] || { echo "WARN missing/empty: $f" >&2; continue; }

  cut -f1-3 "$f" \
    | LC_ALL=C sort -k1,1 -k2,2n \
    | bedtools merge -i - \
    | bedtools intersect -a - -b "$WORK/consensus.bed" -v \
    | awk -v s="$s" 'BEGIN{OFS="\t"}{print $1,$2,$3,s}' \
    > "$WORK/new.bed"

  echo "$s: +$(wc -l < "$WORK/new.bed")" >&2
  cat "$WORK/new.bed" >> "$WORK/consensus.bed"
  LC_ALL=C sort -k1,1 -k2,2n -o "$WORK/consensus.bed" "$WORK/consensus.bed"
done

cp "$WORK/consensus.bed" "$OUT"
rm -rf "$WORK"

wc -l "$OUT"

shCd19_Runx3: +2674
shRunx3_Runx3: +774
shRunx3_Runx1: +204
shCd19_Runx1: +471
terminal_Runx3: +12
late_Runx3: +27
early_Runx3: +0
memory_Runx3: +10
WARN missing/empty: 01_peakEDA/04_peakConsolidate/individual/terminal_Runx1.CPM.stringent.named.good.bed
WARN missing/empty: 01_peakEDA/04_peakConsolidate/individual/late_Runx1.CPM.stringent.named.good.bed
WARN missing/empty: 01_peakEDA/04_peakConsolidate/individual/early_Runx1.CPM.stringent.named.good.bed
memory_Runx1: +0
4172 01_peakEDA/04_peakConsolidate/fullpeak.clean.bed


In [29]:
deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    ${bigWigFiles["memory_Runx3"]} \
    ${bigWigFiles["early_Runx3"]} \
    ${bigWigFiles["late_Runx3"]} \
    ${bigWigFiles["terminal_Runx3"]} \
    ${bigWigFiles["memory_Runx1"]} \
    ${bigWigFiles["early_Runx1"]} \
    ${bigWigFiles["late_Runx1"]} \
    ${bigWigFiles["terminal_Runx1"]} \
    -R \
    01_peakEDA/04_peakConsolidate/fullpeak.clean.bed \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 01_peakEDA/04_peakConsolidate/fullpeak.clean.gz


k=2
deeptools plotHeatmap \
    -m "01_peakEDA/04_peakConsolidate/fullpeak.clean.gz" \
    -out "01_peakEDA/04_peakConsolidate/fullpeak.clean.k${k}.pdf" \
    --kmeans ${k} \
    --sortUsing sum \
    --colorMap RdYlBu_r RdYlBu_r RdYlGn_r RdYlGn_r RdYlBu_r RdYlBu_r RdYlBu_r RdYlBu_r RdYlGn_r RdYlGn_r RdYlGn_r RdYlGn_r \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                    "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                    "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
    --zMin 0 \
    --zMax 1.25 1.25 1.00 1.00 1.20 1.00 1.00 1.00 0.75 0.75 0.75 0.75 \
    --yMax 2.40 2.40 1.20 1.20 1.00 1.00 1.00 1.00 0.55 0.55 0.55 0.55 \
    --outFileSortedRegions "01_peakEDA/04_peakConsolidate/fullpeak.clean.clustered-k${k}.bed"


The following chromosome names did not match between the bigwig files
chromosome	length
              Y	  91744698
*Warning* For clustering nan values have to be replaced by zeros 


In [34]:
cat 01_peakEDA/04_peakConsolidate/individual/* > 01_peakEDA/04_peakConsolidate/individual/all.named.bed

NAMED=01_peakEDA/04_peakConsolidate/individual/all.named.bed
CLUST=01_peakEDA/04_peakConsolidate/fullpeak.clean.clustered-k2.bed
OUT=01_peakEDA/fullpeak.clean.clustered.named.bed

awk 'BEGIN{OFS="\t"}
  # pass 1: coordinate -> source name
  NR==FNR { name[$1"\t"$2"\t"$3] = $4; next }

  # pass 2: clustered file
  /^#/ { next }
  {
    k = $1"\t"$2"\t"$3
    if (k in name) print $1, $2, $3, name[k], $NF
    else         { print $1, $2, $3, "NA", $NF; miss++ }
    n++
  }
  END {
    printf("%d peaks written, %d unmatched\n", n, miss+0) > "/dev/stderr"
  }
' "$NAMED" "$CLUST" \
  | LC_ALL=C sort -k1,1 -k2,2n \
  > "$OUT"

4172 peaks written, 0 unmatched


In [37]:
grep "cluster_1" 01_peakEDA/fullpeak.clean.clustered.named.bed > 01_peakEDA/cluster1.fullpeak.clean.bed
grep "cluster_2" 01_peakEDA/fullpeak.clean.clustered.named.bed > 01_peakEDA/cluster2.fullpeak.clean.bed